# Order SLA — POC

**Design:** `openspec/changes/order-sla/design.md`  
**Queries:** `queries/sla_standard.sql`, `queries/sla_platform.sql`  
**Audience:** BOD — ≤30s scan  

> **Nguyên tắc:** Notebook này chỉ visualize — không transform, filter, hay tính toán business logic.
> MoM delta tính từ 2 rows liền kề trong CSV (presentation logic).
> Toàn bộ SLA logic đã xử lý trong SQL (verified qua MCP).

## 1. Setup

In [ ]:
import pandas as pd
import plotly.graph_objects as go
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

REPORT_NAME = 'order-sla'

## 2. Load Data

In [ ]:
std = pd.read_csv('queries/sla_standard.csv')
plt_ = pd.read_csv('queries/sla_platform.csv')

# Combine — chỉ dùng cột cần thiết cho viz
df = pd.concat([
    std[['month', 'group', 'total_orders', 'tal_pass_rate']],
    plt_[['month', 'group', 'total_orders', 'tal_pass_rate']],
], ignore_index=True).sort_values(['group', 'month'])

print(f'Standard: {len(std)} months | Platform: {len(plt_)} months')
df

## 3. View 1 — Scorecard (tháng hiện tại + MoM delta)

Dùng tháng có đủ data gần nhất: loại tháng hiện tại nếu < 15 ngày đã qua.

In [ ]:
from datetime import date

def get_scorecard(df, group):
    """Lấy current month và previous month cho 1 group."""
    g = df[df['group'] == group].sort_values('month').reset_index(drop=True)
    # Dùng tháng cuối có đủ volume (>= 30 orders)
    g_valid = g[g['total_orders'] >= 30]
    if len(g_valid) < 2:
        return None
    curr = g_valid.iloc[-1]
    prev = g_valid.iloc[-2]
    delta = curr['tal_pass_rate'] - prev['tal_pass_rate']
    return {
        'group': group,
        'month': curr['month'],
        'pass_rate': curr['tal_pass_rate'],
        'total_orders': int(curr['total_orders']),
        'delta': round(delta, 1),
        'prev_month': prev['month'],
        'prev_pass_rate': prev['tal_pass_rate'],
    }

scorecard_std = get_scorecard(df, 'Standard')
scorecard_plt = get_scorecard(df, 'Platform')

for s in [scorecard_std, scorecard_plt]:
    if s:
        arrow = '↑' if s['delta'] >= 0 else '↓'
        print(f"{s['group']:10s} | {s['month']} | {s['pass_rate']:5.1f}%  {arrow}{abs(s['delta']):.1f}% vs {s['prev_month']}  | n={s['total_orders']:,}")

In [ ]:
# Plotly Scorecard
fig_scorecard = go.Figure()

for s in [scorecard_std, scorecard_plt]:
    if not s:
        continue
    delta_str = f"{'▲' if s['delta'] >= 0 else '▼'} {abs(s['delta']):.1f}%"
    color = '#2ecc71' if s['delta'] >= 0 else '#e74c3c'
    footnote = ' (*)' if s['group'] == 'Standard' else ''

    fig_scorecard.add_trace(go.Indicator(
        mode='number+delta',
        value=s['pass_rate'],
        number={'suffix': '%', 'font': {'size': 52}},
        delta={
            'reference': s['prev_pass_rate'],
            'valueformat': '.1f',
            'suffix': '%',
            'increasing': {'color': '#2ecc71'},
            'decreasing': {'color': '#e74c3c'},
        },
        title={'text': f"<b>{s['group']}</b>{footnote}<br><span style='font-size:13px;color:gray'>{s['month']} | SLA {'≤3' if s['group'] == 'Platform' else '≤4'} biz days | n={s['total_orders']:,}</span>"},
        domain={'x': [0, 0.45] if s['group'] == 'Standard' else [0.55, 1], 'y': [0, 1]},
    ))

fig_scorecard.update_layout(
    title={
        'text': 'SLA Pass Rate — Current Month<br><sup>(*) Standard: chỉ tính đơn có tracking data</sup>',
        'font': {'size': 18},
    },
    height=280,
    paper_bgcolor='white',
)
fig_scorecard.show()

## 4. View 2 — Trend (12 tháng)

Chỉ hiển thị tháng có `total_orders >= 30` để tránh nhiễu từ tháng ít data.

In [ ]:
df_trend = df[df['total_orders'] >= 30].copy()

fig_trend = go.Figure()

colors = {'Standard': '#3498db', 'Platform': '#e67e22'}
thresholds = {'Standard': 4, 'Platform': 3}

for group in ['Standard', 'Platform']:
    g = df_trend[df_trend['group'] == group].sort_values('month')
    fig_trend.add_trace(go.Scatter(
        x=g['month'],
        y=g['tal_pass_rate'],
        mode='lines+markers',
        name=f"{group} (SLA ≤{thresholds[group]}d)",
        line={'color': colors[group], 'width': 2},
        marker={'size': 7},
        hovertemplate='%{x}<br>Pass rate: %{y:.1f}%<extra>' + group + '</extra>',
    ))

fig_trend.update_layout(
    title={'text': 'SLA TAL Pass Rate — 12 Tháng', 'font': {'size': 18}},
    xaxis_title='Tháng',
    yaxis_title='TAL Pass Rate (%)',
    yaxis={'range': [0, 105], 'ticksuffix': '%'},
    hovermode='x unified',
    legend={'orientation': 'h', 'yanchor': 'bottom', 'y': 1.02},
    height=420,
    paper_bgcolor='white',
    plot_bgcolor='#f8f9fa',
)
fig_trend.show()

## 5. Notes

- **Findings:** Standard pass rate thấp nhất Dec 2025 (43.5%) — có thể holiday season. Platform data có ý nghĩa từ Feb 2026 trở đi (trước đó < 30 orders/tháng).
- **Filter total_orders >= 30:** Loại tháng ít data tránh nhiễu trên BOD view (Platform pre-Feb 2026 bị loại).
- **MoM delta note:** Platform chỉ có 2 tháng đủ data (Feb + Mar 2026) — delta giữa 2 tháng này.
- **Next steps (Metabase):** 2 queries SQL → 2 cards: Scorecard + Trend line chart.